<a href="https://colab.research.google.com/github/sara-iqbal/financial-etl-pipeline/blob/main/etl_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End Data Pipeline with ML & SQL Analytics
### PySpark ETL · Risk Scoring · SQL Analytics · Dashboard

**Author:** Sara | MSc Data Science | B.Tech Artificial Intelligence & Machine Learning

---

| Step | Task |
|------|------|
| 1 | Install dependencies |
| 2 | Generate synthetic financial transaction data (Faker) |
| 3 | Ingest & explore raw data with PySpark |
| 4 | ETL — clean, validate, transform |
| 5 | Data quality checks |
| 6 | Feature engineering |
| 7 | Train XGBoost risk-scoring model |
| 8 | SQL analytics with pandasql |
| 9 | Visualise results — pipeline for Looker Studio |
| 10 | Export clean CSV for Google Looker Studio |


## Step 1 Install Dependencies

In [1]:
!pip install -q pyspark faker pandasql xgboost plotly scikit-learn pandas numpy
print('All dependencies installed!')

All dependencies installed!


## Step 2 Import Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings, os, random, json
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

# Faker for synthetic data
from faker import Faker

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType,
    TimestampType, BooleanType
)
from pyspark.sql.window import Window

# ML
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, average_precision_score
)
from sklearn.preprocessing import LabelEncoder
import joblib

# SQL
import pandasql as psql

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

print('All imports successful!')

All imports successful!


## Step 3 Generate Synthetic Financial Transaction Data

Generate **50,000 realistic financial transactions** using the Faker library.
The dataset mimics a real banking transaction ledger with:
- Customer demographics and account types
- Transaction amounts, merchants, and categories
- Timestamps across 2 years
- A synthetic fraud/risk label with realistic class imbalance (3% fraud rate)

In [3]:
N_TRANSACTIONS = 50_000
FRAUD_RATE     = 0.03   # 3% fraud — realistic imbalance

CATEGORIES  = ['groceries','electronics','travel','dining','utilities',
                'healthcare','entertainment','fashion','fuel','transfer']
ACCOUNT_TYPES = ['current','savings','premium','business']
COUNTRIES   = ['GB','US','DE','FR','SG','AE','IN','AU','CA','JP']
RISK_COUNTRIES = ['AE','SG','IN']   # slightly elevated risk

def generate_transactions(n):
    rows = []
    start_date = datetime(2022, 1, 1)
    end_date   = datetime(2024, 1, 1)
    date_range = (end_date - start_date).days

    # Pre-generate customer pool (realistic repeat customers)
    n_customers = n // 8
    customers = [
        {
            'customer_id': f'CUST_{i:06d}',
            'age':          random.randint(18, 75),
            'account_type': random.choice(ACCOUNT_TYPES),
            'country':      random.choice(COUNTRIES),
            'credit_score': random.randint(300, 850),
            'account_age_days': random.randint(30, 3650)
        }
        for i in range(n_customers)
    ]

    for i in range(n):
        cust = random.choice(customers)
        cat  = random.choice(CATEGORIES)

        # Amount logic — varies by category and fraud
        is_fraud = random.random() < FRAUD_RATE
        if is_fraud:
            amount = round(random.uniform(500, 8000), 2)
        elif cat == 'travel':
            amount = round(random.uniform(100, 3000), 2)
        elif cat == 'electronics':
            amount = round(random.uniform(50, 2000), 2)
        elif cat in ['groceries','fuel','utilities']:
            amount = round(random.uniform(5, 200), 2)
        else:
            amount = round(random.uniform(10, 500), 2)

        ts = start_date + timedelta(
            days=random.randint(0, date_range),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59)
        )

        rows.append({
            'transaction_id':   f'TXN_{i:08d}',
            'customer_id':      cust['customer_id'],
            'timestamp':        ts.strftime('%Y-%m-%d %H:%M:%S'),
            'amount':           amount,
            'currency':         'GBP' if cust['country']=='GB' else 'USD',
            'category':         cat,
            'merchant':         fake.company(),
            'merchant_country': random.choice(COUNTRIES),
            'account_type':     cust['account_type'],
            'customer_age':     cust['age'],
            'credit_score':     cust['credit_score'],
            'account_age_days': cust['account_age_days'],
            'customer_country': cust['country'],
            'channel':          random.choice(['mobile','web','atm','branch']),
            'is_international': cust['country'] != random.choice(COUNTRIES),
            'is_fraud':         int(is_fraud)
        })

        # Inject realistic data quality issues (~2% rate)
        if random.random() < 0.02:
            field = random.choice(['amount','merchant','category'])
            if field == 'amount':   rows[-1]['amount']   = None
            if field == 'merchant': rows[-1]['merchant'] = None
            if field == 'category': rows[-1]['category'] = 'unknown'

        # Inject some negative amounts (refunds)
        if random.random() < 0.015:
            rows[-1]['amount'] = -abs(rows[-1].get('amount') or 10)

    return pd.DataFrame(rows)

print(f'Generating {N_TRANSACTIONS:,} synthetic transactions...')
raw_df = generate_transactions(N_TRANSACTIONS)

print(f'✅ Generated {len(raw_df):,} transactions')
print(f'   Fraud rate : {raw_df["is_fraud"].mean():.2%}')
print(f'   Nulls      : {raw_df.isnull().sum().sum():,}')
print(f'   Date range : {raw_df["timestamp"].min()} → {raw_df["timestamp"].max()}')
raw_df.head()

Generating 50,000 synthetic transactions...
✅ Generated 50,000 transactions
   Fraud rate : 3.07%
   Nulls      : 640
   Date range : 2022-01-01 00:08:00 → 2024-01-01 22:58:00


,transaction_id,customer_id,timestamp,amount,currency,category,merchant,merchant_country,account_type,customer_age,credit_score,account_age_days,customer_country,channel,is_international,is_fraud
0,TXN_00000000,CUST_003423,2022-04-21 13:29:00,1677.07,USD,travel,"Rodriguez, Figueroa and Sanchez",FR,current,44,342,2158,CA,web,True,0
1,TXN_00000001,CUST_001150,2023-01-19 13:46:00,1669.39,GBP,electronics,Doyle Ltd,DE,premium,23,361,1486,GB,branch,True,0
2,TXN_00000002,CUST_003475,2023-06-08 20:28:00,107.52,USD,groceries,"Mcclain, Miller and Henderson",IN,business,29,372,628,CA,web,True,0
3,TXN_00000003,CUST_003472,2023-03-18 07:01:00,305.72,USD,transfer,Davis and Sons,FR,current,35,440,1338,SG,atm,True,0
4,TXN_00000004,CUST_000485,2023-09-23 18:34:00,143.89,USD,utilities,"Guzman, Hoffman and Baldwin",DE,premium,25,526,927,AE,web,True,0


## Step 4 Ingest Raw Data into PySpark

We initialise a local Spark session and load the raw Pandas DataFrame into a
Spark DataFrame — simulating reading from a data lake or HDFS in production.

In [ ]:
# Initialise Spark session
spark = SparkSession.builder \
    .appName('FinancialETLPipeline') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'Spark version : {spark.version}')

# Save raw data to CSV (simulates landing zone / data lake)
raw_df.to_csv('transactions_raw.csv', index=False)
print('Raw CSV saved to transactions_raw.csv')

# Define explicit schema
schema = StructType([
    StructField('transaction_id',   StringType(),  True),
    StructField('customer_id',      StringType(),  True),
    StructField('timestamp',        StringType(),  True),
    StructField('amount',           DoubleType(),  True),
    StructField('currency',         StringType(),  True),
    StructField('category',         StringType(),  True),
    StructField('merchant',         StringType(),  True),
    StructField('merchant_country', StringType(),  True),
    StructField('account_type',     StringType(),  True),
    StructField('customer_age',     IntegerType(), True),
    StructField('credit_score',     IntegerType(), True),
    StructField('account_age_days', IntegerType(), True),
    StructField('customer_country', StringType(),  True),
    StructField('channel',          StringType(),  True),
    StructField('is_international', BooleanType(), True),
    StructField('is_fraud',         IntegerType(), True),
])

# Load into Spark
sdf = spark.read.csv(
    'transactions_raw.csv',
    header=True,
    schema=schema
)

print(f'\nSpark DataFrame loaded:')
print(f'  Rows       : {sdf.count():,}')
print(f'  Columns    : {len(sdf.columns)}')
print(f'  Partitions : {sdf.rdd.getNumPartitions()}')
sdf.printSchema()

Spark version : 4.0.2
Raw CSV saved to transactions_raw.csv

Spark DataFrame loaded:


## Step 5 ETL: Extract, Transform, Load

In [ ]:
print('Running ETL pipeline...')
print('='*50)

#  EXTRACT: baseline counts
raw_count = sdf.count()
print(f'[EXTRACT] Raw rows: {raw_count:,}')

#  TRANSFORM 1: parse timestamp
sdf_t = sdf.withColumn(
    'timestamp',
    F.to_timestamp('timestamp', 'yyyy-MM-dd HH:mm:ss')
)

# TRANSFORM 2: extract date parts
sdf_t = sdf_t \
    .withColumn('year',        F.year('timestamp')) \
    .withColumn('month',       F.month('timestamp')) \
    .withColumn('day_of_week', F.dayofweek('timestamp')) \
    .withColumn('hour',        F.hour('timestamp')) \
    .withColumn('is_weekend',  (F.dayofweek('timestamp').isin([1,7])).cast(IntegerType()))

# TRANSFORM 3: clean amount
null_amounts = sdf_t.filter(F.col('amount').isNull()).count()
neg_amounts  = sdf_t.filter(F.col('amount') < 0).count()
print(f'[CLEAN]   Null amounts   : {null_amounts:,}')
print(f'[CLEAN]   Negative amounts (refunds): {neg_amounts:,}')

# Fill null amounts with category median (using approx)
sdf_t = sdf_t.withColumn(
    'amount',
    F.when(F.col('amount').isNull(), F.lit(50.0))
     .otherwise(F.col('amount'))
)
# Flag refunds, keep absolute value
sdf_t = sdf_t \
    .withColumn('is_refund', (F.col('amount') < 0).cast(IntegerType())) \
    .withColumn('amount', F.abs(F.col('amount')))

# TRANSFORM 4: standardise category
sdf_t = sdf_t.withColumn(
    'category',
    F.when(F.col('category').isin(['unknown', None]), F.lit('other'))
     .otherwise(F.lower(F.col('category')))
)

# TRANSFORM 5: fill null merchant
sdf_t = sdf_t.withColumn(
    'merchant',
    F.when(F.col('merchant').isNull(), F.lit('UNKNOWN_MERCHANT'))
     .otherwise(F.col('merchant'))
)

# TRANSFORM 6: amount buckets
sdf_t = sdf_t.withColumn(
    'amount_bucket',
    F.when(F.col('amount') < 50,    F.lit('micro'))
     .when(F.col('amount') < 200,   F.lit('small'))
     .when(F.col('amount') < 1000,  F.lit('medium'))
     .when(F.col('amount') < 5000,  F.lit('large'))
     .otherwise(F.lit('xlarge'))
)

# TRANSFORM 7: is_high_risk_country
sdf_t = sdf_t.withColumn(
    'is_high_risk_country',
    F.col('merchant_country').isin(RISK_COUNTRIES).cast(IntegerType())
)

# LOAD: final cleaned DataFrame
clean_count = sdf_t.count()
print(f'[LOAD]    Clean rows: {clean_count:,}')
print(f'[LOAD]    Dropped   : {raw_count - clean_count:,}')
print(f'\n✅ ETL complete!')
sdf_t.show(5, truncate=True)

## Step 6 Data Quality Checks

In [ ]:
print('Running data quality checks...')
print('='*55)

checks = {}

# 1. Null check — no nulls allowed in key fields
key_fields = ['transaction_id','customer_id','amount','timestamp','category']
for col in key_fields:
    n = sdf_t.filter(F.col(col).isNull()).count()
    status = '✅ PASS' if n == 0 else f'❌ FAIL ({n} nulls)'
    checks[f'null_check_{col}'] = status
    print(f'  Null check [{col:<20}] : {status}')

# 2. Amount range — all amounts must be positive
neg = sdf_t.filter(F.col('amount') <= 0).count()
status = '✅ PASS' if neg == 0 else f'❌ FAIL ({neg} non-positive)'
checks['amount_positive'] = status
print(f'  Amount positive check      : {status}')

# 3. Duplicate transaction IDs
total = sdf_t.count()
unique = sdf_t.select('transaction_id').distinct().count()
status = '✅ PASS' if total == unique else f'❌ FAIL ({total-unique} duplicates)'
checks['duplicate_txn_ids'] = status
print(f'  Duplicate txn ID check     : {status}')

# 4. Valid categories
valid_cats = CATEGORIES + ['other']
invalid = sdf_t.filter(~F.col('category').isin(valid_cats)).count()
status = '✅ PASS' if invalid == 0 else f'❌ FAIL ({invalid} invalid)'
checks['valid_categories'] = status
print(f'  Valid category check       : {status}')

# 5. Timestamp range
ts_stats = sdf_t.agg(
    F.min('timestamp').alias('min_ts'),
    F.max('timestamp').alias('max_ts')
).collect()[0]
print(f'  Timestamp range            : {ts_stats["min_ts"]} → {ts_stats["max_ts"]}')
print(f'\n✅ Data quality checks complete!')

# 6. Summary stats
print('\nAmount statistics:')
sdf_t.select('amount').describe().show()

## Step 7 — Feature Engineering for Risk Scoring

In [ ]:
print('Engineering features...')

# Customer-level aggregates (rolling window features)
w_customer = Window.partitionBy('customer_id').orderBy('timestamp')
w_customer_30d = Window.partitionBy('customer_id') \
    .orderBy(F.col('timestamp').cast('long')) \
    .rangeBetween(-30*86400, 0)

sdf_feat = sdf_t \
    .withColumn('txn_count_total',
        F.count('transaction_id').over(w_customer)) \
    .withColumn('avg_amount_customer',
        F.avg('amount').over(w_customer)) \
    .withColumn('max_amount_customer',
        F.max('amount').over(w_customer)) \
    .withColumn('amount_vs_avg',
        F.col('amount') / (F.col('avg_amount_customer') + F.lit(1.0))) \
    .withColumn('txn_rank',
        F.rank().over(w_customer)) \
    .withColumn('is_night_txn',
        ((F.col('hour') < 6) | (F.col('hour') > 22)).cast(IntegerType()))

# Convert to Pandas for ML
print('Converting to Pandas for ML...')
pdf = sdf_feat.toPandas()
pdf['timestamp'] = pd.to_datetime(pdf['timestamp'])

# Encode categoricals
le = LabelEncoder()
for col in ['category','account_type','channel','customer_country',
            'merchant_country','currency','amount_bucket']:
    pdf[col + '_enc'] = le.fit_transform(pdf[col].astype(str))

FEATURE_COLS = [
    'amount', 'customer_age', 'credit_score', 'account_age_days',
    'is_international', 'is_weekend', 'is_night_txn', 'is_refund',
    'is_high_risk_country', 'hour', 'day_of_week', 'month',
    'amount_vs_avg', 'txn_count_total', 'avg_amount_customer',
    'max_amount_customer',
    'category_enc', 'account_type_enc', 'channel_enc',
    'merchant_country_enc', 'amount_bucket_enc'
]

X = pdf[FEATURE_COLS].fillna(0)
y = pdf['is_fraud']

print(f'\n Feature engineering complete!')
print(f'   Features  : {len(FEATURE_COLS)}')
print(f'   Samples   : {len(X):,}')
print(f'   Fraud rate: {y.mean():.2%}')

## Step 8 Train XGBoost Risk-Scoring Model

In [ ]:
#  Feature importance
fi = pd.Series(model.feature_importances_, index=FEATURE_COLS)
fi = fi.sort_values(ascending=False).head(15)

# Fix: name the columns explicitly after reset_index()
fi_df = fi.reset_index()
fi_df.columns = ['feature', 'importance']

fig = px.bar(
    fi_df,
    x='importance', y='feature',
    orientation='h',
    title='Top 15 Feature Importances — XGBoost Risk Model',
    color='importance',
    color_continuous_scale=[
        [0.0, '#e8f4f0'], [0.5, '#7a9e87'], [1.0, '#4a7a5e']
    ]
)
fig.update_layout(
    template='plotly_white', height=480,
    yaxis={'categoryorder': 'total ascending'}
)
fig.show()

print('\nModel training complete!')
joblib.dump(model, 'risk_model.pkl')
joblib.dump(FEATURE_COLS, 'feature_cols.pkl')
print('Model saved: risk_model.pkl')

## Step 9 SQL Analytics with pandasql

In [ ]:
# Move the copy to AFTER Step 8 adds risk_score
# Check risk_score exists before running SQL
print('Columns in pdf:', [c for c in pdf.columns if 'risk' in c])

# Re-copy now that risk_score is attached
transactions = pdf.copy()

# Verify the column is there
assert 'risk_score' in transactions.columns, "risk_score missing — run Step 8 first!"
print(f'✅ risk_score column found — {len(transactions):,} rows ready for SQL')
print('='*55)
# ── Q1: Total transactions and fraud by category ──────────
q1 = sql("""
    SELECT
        category,
        COUNT(*)                                        AS total_txns,
        SUM(is_fraud)                                   AS fraud_count,
        ROUND(AVG(is_fraud) * 100, 2)                  AS fraud_rate_pct,
        ROUND(AVG(amount), 2)                           AS avg_amount,
        ROUND(SUM(amount), 2)                           AS total_volume
    FROM transactions
    GROUP BY category
    ORDER BY fraud_rate_pct DESC
""")
print('\n📊 Q1: Fraud Rate by Category')
print(q1.to_string(index=False))

# ── Q2: High-risk transactions (score > 0.6) ──────────────
q2 = sql("""
    SELECT
        account_type,
        COUNT(*)                                        AS flagged_txns,
        ROUND(AVG(risk_score) * 100, 2)                AS avg_risk_score_pct,
        ROUND(SUM(amount), 2)                           AS total_flagged_value
    FROM transactions
    WHERE risk_score > 0.6
    GROUP BY account_type
    ORDER BY flagged_txns DESC
""")
print('\n📊 Q2: High-Risk Transactions by Account Type')
print(q2.to_string(index=False))

# ── Q3: Monthly transaction volume and fraud ──────────────
q3 = sql("""
    SELECT
        year,
        month,
        COUNT(*)                                        AS txn_count,
        ROUND(SUM(amount), 2)                           AS total_volume,
        SUM(is_fraud)                                   AS fraud_count,
        ROUND(AVG(risk_score) * 100, 2)                AS avg_risk_pct
    FROM transactions
    GROUP BY year, month
    ORDER BY year, month
""")
print('\n📊 Q3: Monthly Volume & Fraud Trend')
print(q3.head(12).to_string(index=False))

# ── Q4: Top 10 riskiest customers ────────────────────────
q4 = sql("""
    SELECT
        customer_id,
        COUNT(*)                                        AS txn_count,
        ROUND(AVG(risk_score) * 100, 2)                AS avg_risk_pct,
        ROUND(SUM(amount), 2)                           AS total_spend,
        SUM(is_fraud)                                   AS confirmed_fraud
    FROM transactions
    GROUP BY customer_id
    ORDER BY avg_risk_pct DESC
    LIMIT 10
""")
print('\n📊 Q4: Top 10 Riskiest Customers')
print(q4.to_string(index=False))

print('\n✅ SQL analytics complete!')

## Step 10 Visualise Pipeline Results

In [ ]:
#  Plot 1: Transaction volume + fraud by month
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Monthly Transaction Volume',
        'Risk Score Distribution',
        'Fraud Rate by Category',
        'Transaction Amount by Risk Level'
    )
)

# Monthly volume
q3['period'] = q3['year'].astype(str) + '-' + q3['month'].astype(str).str.zfill(2)
fig.add_trace(go.Bar(
    x=q3['period'], y=q3['total_volume'],
    name='Volume', marker_color='#7a9e87'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=q3['period'], y=q3['fraud_count'],
    name='Fraud', mode='lines+markers',
    line=dict(color='#fc8181', width=2), yaxis='y2'
), row=1, col=1)

# Risk score histogram
fig.add_trace(go.Histogram(
    x=pdf['risk_score'], nbinsx=50,
    name='Risk Score',
    marker_color='#63b3ed', opacity=0.8
), row=1, col=2)

# Fraud by category
fig.add_trace(go.Bar(
    x=q1['category'], y=q1['fraud_rate_pct'],
    name='Fraud Rate %',
    marker_color='#fc8181'
), row=2, col=1)

# Amount box by risk level
for rl, col in [('Low','#48bb78'),('Medium','#f6ad55'),('High','#fc8181')]:
    sub = pdf[pdf['risk_label'] == rl]['amount']
    fig.add_trace(go.Box(
        y=sub.clip(0, 2000), name=rl,
        marker_color=col, showlegend=False
    ), row=2, col=2)

fig.update_layout(
    height=700, template='plotly_white',
    title='ETL Pipeline — Analytics Dashboard',
    showlegend=False
)
fig.show()

# Plot 2: Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig2 = px.imshow(
    cm, text_auto=True,
    x=['Pred: Legit','Pred: Fraud'],
    y=['True: Legit','True: Fraud'],
    color_continuous_scale=[[0,'#f7f4f0'],[0.5,'#a8c4b0'],[1,'#4a7a5e']],
    title=f'XGBoost Confusion Matrix  |  ROC-AUC: {roc_auc:.4f}'
)
fig2.update_layout(template='plotly_white', height=380)
fig2.show()

print(' Visualisations complete!')

In [ ]:
# Columns to export for Looker Studio
export_cols = [
    'transaction_id','customer_id','timestamp','amount','currency',
    'category','merchant_country','account_type','customer_age',
    'credit_score','account_age_days','customer_country','channel',
    'is_international','is_weekend','is_night_txn','is_refund',
    'is_high_risk_country','hour','month','year','amount_bucket',
    'risk_score','risk_label','is_fraud'
]

export_df = pdf[export_cols].copy()
export_df['risk_score'] = export_df['risk_score'].round(4)
export_df['timestamp']  = export_df['timestamp'].astype(str)

export_df.to_csv('transactions_enriched.csv', index=False)

print('Exported: transactions_enriched.csv')
print(f'   Rows    : {len(export_df):,}')
print(f'   Columns : {len(export_cols)}')
print(f'   Size    : {os.path.getsize("transactions_enriched.csv")/1e6:.1f} MB')
print()

# Also export SQL summary tables for Looker
q1.to_csv('summary_by_category.csv', index=False)
q2.to_csv('summary_high_risk.csv',   index=False)
q3.to_csv('summary_monthly.csv',     index=False)

print('Summary tables exported:')
print('  summary_by_category.csv')
print('  summary_high_risk.csv')
print('  summary_monthly.csv')
print()
print('='*55)
print('To download all files from Colab:')
print('  from google.colab import files')
print('  files.download("transactions_enriched.csv")')
print('  files.download("risk_model.pkl")')

In [ ]:
# Run this to download all output files to your computer
from google.colab import files
files.download('transactions_enriched.csv')
files.download('summary_by_category.csv')
files.download('summary_monthly.csv')
files.download('risk_model.pkl')
print('All files downloaded!')

---
## ✅ Project Complete!

| Deliverable | Status |
|---|---|
| 50,000 synthetic transactions generated with Faker | ✅ |
| PySpark ETL pipeline with 7 transforms | ✅ |
| Data quality checks (nulls, duplicates, ranges) | ✅ |
| 21 engineered features including window functions | ✅ |
| XGBoost risk-scoring model trained | ✅ |
| SQL analytics — 4 business queries with pandasql | ✅ |
| Visual dashboard with Plotly | ✅ |
| Enriched CSV exported for Looker Studio | ✅ |

**Next step:** Upload `transactions_enriched.csv` to Google Looker Studio for the live public dashboard link.